# Blackjack Session Analysis
## Betting Strategies, Card Counting, and the Long-Term Reality

The strategy notebook asked which *decisions* are correct. This one asks what happens to your *money*. A single hand is noise; a session of 1,000 hands is signal — and across **10,000 sessions per configuration** (each 1,000 hands, $1,000 starting bankroll, $10 base bet, fixed seed), patterns emerge that are invisible at the hand level: how variance compounds, why popular betting systems fail, and the exact conditions under which card counting turns a losing game into a winning one.

Nine configurations, always compared on net profit in dollars. Three questions organise the notebook:

1. **What does the house edge actually look like over a real session** — not as a percentage, but as a distribution of outcomes?
2. **Do betting systems change anything?** Martingale, Anti-Martingale, flat — strong beliefs surround them.
3. **Does card counting work** — under what conditions, by how much, and at what risk?

*Hand-level strategy analysis is in `strategy_comparison.ipynb`. Rules throughout: 6 decks (or 1 where noted), dealer stands on soft 17, blackjack pays 3:2.*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titleweight": "bold"})

RUNS_DIR = "../data/runs"
BASE_BET = 10
START_BANKROLL = 1000

SESSION_RUNS = {
    "Basic + Flat":               ("sess_basic_flat",          "#2196F3"),
    "Basic + Martingale":         ("sess_basic_martingale",    "#F44336"),
    "Basic + Anti-Martingale":    ("sess_basic_anti",          "#FF9800"),
    "Basic + Count (no count)":   ("sess_basic_count_nocount", "#9E9E9E"),
    "Basic + Count + HiLo 6D":    ("sess_basic_count_hilo_6d", "#4CAF50"),
    "Basic + Count + HiLo 1D":    ("sess_basic_count_hilo_1d", "#009688"),
    "Random + Flat":              ("sess_random_flat",         "#795548"),
    "Counting + HiLo 6D":         ("sess_counting_hilo_6d",    "#3F51B5"),
    "Counting + HiLo 1D":         ("sess_counting_hilo_1d",    "#E91E63"),
}
COLORS = {lbl: c for lbl, (r, c) in SESSION_RUNS.items()}

# Decision logs are large -> load only for the counting runs and only needed columns.
COUNTING = ["Basic + Count + HiLo 6D", "Basic + Count + HiLo 1D",
            "Counting + HiLo 6D", "Counting + HiLo 1D"]
USE_DEC = ["session_id", "decision_index", "player_value", "player_is_soft",
           "dealer_upcard", "true_count", "bet_size", "is_win", "action", "can_split"]
DEC_DT = {"decision_index": "int16", "player_value": "int16",
          "dealer_upcard": "int16", "player_is_soft": "bool", "true_count": "float32",
          "bet_size": "float32", "is_win": "int8", "action": "category", "can_split": "bool"}

missing = [lbl for lbl, (run, _) in SESSION_RUNS.items()
           if not os.path.exists(f"{RUNS_DIR}/{run}/sessions.csv")]
if missing:
    raise FileNotFoundError(f"Missing session runs: {missing}. Re-run regenerate_data.py, "
                            f"then RESTART the kernel and Run All.")

sessions, decisions = {}, {}
for lbl, (run, _) in SESSION_RUNS.items():
    sessions[lbl] = pd.read_csv(f"{RUNS_DIR}/{run}/sessions.csv")
for lbl in COUNTING:
    run = SESSION_RUNS[lbl][0]
    decisions[lbl] = pd.read_csv(f"{RUNS_DIR}/{run}/decisions.csv", usecols=USE_DEC, dtype=DEC_DT)

def sess_metrics(s):
    return dict(sessions=len(s), net=s.net_profit.mean(), net_std=s.net_profit.std(),
                bust=s.went_bust.mean() * 100, win=s.win_rate.mean() * 100,
                profitable=(s.net_profit > 0).mean() * 100)
summary = pd.DataFrame({lbl: sess_metrics(s) for lbl, s in sessions.items()}).T

print(f"{'Strategy':<26}{'Sessions':>9}{'Net':>9}{'Ruin%':>7}{'Prof%':>7}")
print("-" * 58)
for lbl in SESSION_RUNS:
    r = summary.loc[lbl]
    print(f"{lbl:<26}{int(r.sessions):>9,}{r.net:>+9.2f}{r.bust:>7.1f}{r.profitable:>7.1f}")

## Data Overview

Nine configurations, **10,000 sessions each**, 1,000 hands per session, $1,000 starting bankroll and a $10 base bet throughout. The table is the whole notebook in miniature — read it once now and the rest is explanation.

Two anchors first. **Random + Flat** loses ~$1,003 of a $1,000 bankroll and goes bankrupt 100% of the time: with no strategy, going broke inside a session is a certainty. **Basic + Flat** loses just $45 — the 0.45% house edge from the strategy notebook, now expressed as dollars over a session. Everything between those poles is the interesting part.

One methodological note: bet sizes differ across configurations (counting strategies raise their bets at favourable counts), so **raw return percentages are not comparable across rows** — every comparison here uses net profit in dollars. The next-but-one section makes the reason concrete.

Read the columns carefully, because two of them are easy to confuse: **Ruin Rate** is the share of sessions that went *bankrupt* (bankroll hit $0) — distinct from a hand “bust” over 21 in the strategy notebook; **Hand Win %** is the average share of individual *hands* won, which barely moves because every strategy here plays the same basic strategy; and **Profitable %** is the share of *sessions* that finished in profit. Win rate and profitability are different questions — a strategy can win a minority of hands and still finish ahead by betting more on the right ones.

In [ ]:
headers = ["Strategy", "Sessions", "Avg Net", "Ruin Rate", "Hand Win %", "Profitable %"]
rows = [[lbl, f"{int(summary.loc[lbl,'sessions']):,}", f"${summary.loc[lbl,'net']:+.2f}",
         f"{summary.loc[lbl,'bust']:.1f}%", f"{summary.loc[lbl,'win']:.1f}%",
         f"{summary.loc[lbl,'profitable']:.1f}%"] for lbl in SESSION_RUNS]

fig, ax = plt.subplots(figsize=(14, 4.5)); ax.axis("off")
tbl = ax.table(cellText=rows, colLabels=headers, cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.9)
for j in range(len(headers)):
    tbl[0, j].set_facecolor("#2C3E50"); tbl[0, j].set_text_props(color="white", fontweight="bold")
for i, lbl in enumerate(SESSION_RUNS):
    for j in range(len(headers)):
        tbl[i + 1, j].set_facecolor("#FDFEFE")
        if j == 2: tbl[i + 1, j].set_text_props(fontweight="bold")
plt.title("Session Analysis Overview", fontsize=12, fontweight="bold", pad=15)
plt.tight_layout(); plt.show()

## The House Edge Over a Real Session

Start with the cleanest case: optimal play, flat $10 bets, no counting. What does a 0.45% edge feel like across 1,000 hands?

In [ ]:
flat = sessions["Basic + Flat"]
pct_prof = (flat.net_profit > 0).mean() * 100
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(flat.ending_bankroll, bins=60, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].axvline(START_BANKROLL, color="black", ls="--", lw=1.5, label=f"Start (${START_BANKROLL:,})")
axes[0].axvline(flat.ending_bankroll.mean(), color="red", lw=1.5,
                label=f"Mean (${flat.ending_bankroll.mean():.0f})")
axes[0].set_title("Ending Bankroll — Basic + Flat"); axes[0].set_xlabel("Ending Bankroll ($)")
axes[0].set_ylabel("Sessions"); axes[0].legend()

axes[1].hist(flat.net_profit, bins=60, color="#2196F3", edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="black", ls="--", lw=1.5, label="Break even")
axes[1].axvline(flat.net_profit.mean(), color="red", lw=1.5, label=f"Mean (${flat.net_profit.mean():.0f})")
axes[1].set_title(f"Net Profit — {pct_prof:.1f}% of sessions profitable")
axes[1].set_xlabel("Net Profit ($)"); axes[1].set_ylabel("Sessions")
stats = (f"Mean: ${flat.net_profit.mean():.0f}\nStd: ${flat.net_profit.std():.0f}\n"
         f"Min: ${flat.net_profit.min():.0f}\nMax: ${flat.net_profit.max():.0f}\n"
         f"Profitable: {pct_prof:.1f}%")
axes[1].text(0.97, 0.97, stats, transform=axes[1].transAxes, fontsize=9, va="top", ha="right",
             bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))
axes[1].legend(loc="upper left")
plt.suptitle("The House Edge Over 1,000 Hands — Basic Strategy + Flat Betting", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

The average session ends down **$45** — the house edge, ground out over 1,000 hands. But the average is almost the least interesting number here. The standard deviation is **$365**: individual sessions run from about **−$1,015 to +$1,375**, and **44.8% finish in profit** despite a guaranteed-negative expectation.

That is the psychology of the casino floor in one histogram. Nearly half of all sessions win money — more than enough to convince a player their system works — yet the distribution is centered just left of zero and stays there forever. Short-term results are not evidence of anything: a player up $300 and a player down $400 are both reading noise. Only the aggregate of thousands of sessions reveals the edge. (The small spike at the far left is the ~1% of sessions that go bankrupt outright.)

## Betting Strategies: Do They Change Outcomes?

If you can't change the cards, can you change your fortunes by changing your *bets*? Martingale (double after every loss), Anti-Martingale (double after every win), and a counting-style variable bet are the usual candidates. Their net-profit distributions tell the story the averages hide.

In [ ]:
bet_strats = ["Basic + Flat", "Basic + Martingale", "Basic + Anti-Martingale", "Basic + Count (no count)"]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, lbl in zip(axes.flatten(), bet_strats):
    s = sessions[lbl]
    mean_net = s.net_profit.mean(); bust = s.went_bust.mean() * 100
    prof = (s.net_profit > 0).mean() * 100
    ax.hist(s.net_profit, bins=60, color=COLORS[lbl], edgecolor="white", alpha=0.85)
    ax.axvline(0, color="black", ls="--", lw=1.5)
    ax.axvline(mean_net, color="darkred", lw=1.5)
    stats = (f"Mean: ${mean_net:.0f}\nStd: ${s.net_profit.std():.0f}\n"
             f"Ruin: {bust:.1f}%\nProfitable: {prof:.1f}%")
    ax.text(0.97, 0.97, stats, transform=ax.transAxes, fontsize=9, va="top", ha="right",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))
    ax.set_title(lbl); ax.set_xlabel("Net Profit ($)"); ax.set_ylabel("Sessions")
plt.suptitle("Betting Strategy Comparison — Net Profit Distribution (Basic Strategy)",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

The means are nearly identical to flat betting — Martingale −$48, Anti-Martingale −$76 — but the *shapes* are not. Both progressive systems are violently **bimodal**: a towering spike at −$1,000 (the bankroll wiped out) beside a broad hump of winning sessions.

**Martingale** goes bankrupt **40.1%** of the time. The doubling logic guarantees that a long enough losing streak — which arrives often in 1,000 hands — exceeds the bankroll before a win can recover it. Survive, and you book a small profit; that's why 47.5% of sessions finish green even as two in five end in ruin. **Anti-Martingale** is worse on average (−$76): riding winning streaks feels good until the streak ends and the doubled bet is gone.

The control case is the quiet one. **Basic + Count (no count)** — the counting-style variable bet with the counting *information switched off* — is **identical to flat betting down to the dollar** (−$45, 1.0% ruin, same distribution). Varying bet size by itself does nothing. Hold onto that: when counting works, it is the *information* doing the work, never the bet pattern.

## Reading Returns Honestly: Bankroll Return vs. Edge

The counting strategies are about to post big positive returns, so it's worth being precise about what "return" means — because two reasonable definitions tell very different stories.

In [ ]:
def avg_wagered(lbl):
    if lbl in decisions:
        return decisions[lbl].groupby("session_id").bet_size.sum().mean()
    return sessions[lbl].hands_played.mean() * BASE_BET  # exact for flat betting

roi_strats = ["Basic + Flat", "Basic + Count + HiLo 6D", "Basic + Count + HiLo 1D",
              "Counting + HiLo 6D", "Counting + HiLo 1D"]
short = ["Basic\nFlat", "Basic+Count\n6D", "Basic+Count\n1D", "Counting\n6D", "Counting\n1D"]
cols = [COLORS[l] for l in roi_strats]
net  = [sessions[l].net_profit.mean() for l in roi_strats]
wag  = [avg_wagered(l) for l in roi_strats]
edge = [n / w * 100 for n, w in zip(net, wag)]   # net profit per dollar actually wagered

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, vals, title, ylab, fmt, zero in [
    (axes[0], net,  "Net Profit ($)\nthe bottom line",                          "Net Profit ($)", "${:.0f}",  True),
    (axes[1], wag,  "Avg Total Wagered ($)\nhow much is put at risk",            "Wagered ($)",    "${:,.0f}", False),
    (axes[2], edge, "Edge per $ Wagered (%)\nthe leverage-free, comparable edge", "Edge (%)",      "{:+.2f}%", True)]:
    bars = ax.bar(short, vals, color=cols)
    if zero:
        ax.axhline(0, color="black", ls="--", lw=1)
    ax.set_title(title); ax.set_ylabel(ylab)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), fmt.format(v),
                ha="center", va="bottom" if v >= 0 else "top", fontsize=9, fontweight="bold")
plt.suptitle("Net Profit Is Real; Edge per Dollar Wagered Is the Comparable Number",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

The **net profit** panel is the honest bottom line: Counting 1D finishes **+$314**, both single-deck strategies clearly positive, everything else below zero. The tempting next step is to quote that as a *return* — **+31% on the $1,000 bankroll** (the `roi` column) — but that figure silently rewards betting more, so it can't fairly compare strategies that don't bet alike.

The middle panel shows the catch: counting pushes **~$24,000** through the table versus **~$10,000** for flat, because it raises bets at favourable counts. Dividing profit by money *actually wagered* gives the **edge per dollar** (right panel) — the one return that is comparable across different bet spreads:

| Strategy | Net | Wagered | Edge per $ wagered |
|---|---:|---:|---:|
| Basic + Flat | −$45 | $9,986 | **−0.45%** |
| Basic + Count 6D | −$12 | $18,180 | −0.07% |
| Counting 6D | −$5 | $17,983 | −0.03% |
| Basic + Count 1D | +$216 | $24,187 | +0.89% |
| Counting 1D | +$314 | $23,814 | **+1.32%** |

So the spectacular "+31%" is really a **+1.3% edge per hand, leveraged into a large bankroll swing by betting 2.4× more.** Both numbers are true and both matter — but reading the bankroll return as if it were the edge would massively overstate how strong counting is. (Reassuringly, flat betting's edge-per-wagered, −0.45%, lands exactly on the hand-level house edge — an internal check that the session engine and the hand engine agree.)

But profit is only half the verdict. Edge-per-dollar says whether a strategy *makes* money — not whether it *survives*, which is a separate question. The single-deck counters that top this chart are also, as the risk-of-ruin section will show, the ones most likely to go bankrupt (20%+ ruin vs flat's 1%): the wide bet spread that earns the edge is exactly what drives the variance, so here more edge is bought with more risk, not for free. One genuine exception hides in the numbers, though — on a single deck, adding index plays (full Counting vs count-betting alone) lifts profit from +$216 to +$314 *and* nudges ruin **down** from 22.1% to 20.4%. Both use the same bet spread, so the variance is identical; the extra profit comes purely from sharper playing, which adds value without widening the bets that create the danger. More profit at slightly less risk is possible — but only when the gain comes from better decisions, not bigger bets.

## Card Counting: Does It Actually Work?

The progression below isolates one variable at a time — bet variation, then information, then index plays, then deck count — so each contribution is visible on its own.

*Methodology note: penetration — how deep the shoe is dealt before reshuffling — is held at **50% for both the 6-deck and single-deck counting runs**, so the 6-deck-vs-single-deck comparison isolates deck count alone instead of confounding it with penetration. Non-counting runs use a realistic 75%; without a counter, penetration is irrelevant, which is exactly why the count-without-counting control matches flat betting to the dollar. (50% is the deepest a single deck can be dealt while still guaranteeing a full hand at every start.)*

In [ ]:
prog = ["Basic + Flat", "Basic + Count (no count)", "Basic + Count + HiLo 6D",
        "Counting + HiLo 6D", "Basic + Count + HiLo 1D", "Counting + HiLo 1D"]
short = ["Basic\nFlat", "Count Bet\nNo Count", "Basic+Count\n6D", "Counting\n6D", "Basic+Count\n1D", "Counting\n1D"]
cols = [COLORS[l] for l in prog]
net = [sessions[l].net_profit.mean() for l in prog]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bars = axes[0].bar(short, net, color=cols)
axes[0].axhline(0, color="black", ls="--", lw=1.5)
axes[0].set_title("Average Net Profit — Counting Progression"); axes[0].set_ylabel("Net Profit ($)")
for b, v in zip(bars, net):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height(), f"${v:+.0f}",
                 ha="center", va="bottom" if v >= 0 else "top", fontsize=9, fontweight="bold")

for lbl in prog[2:]:
    axes[1].hist(sessions[lbl].net_profit, bins=60, alpha=0.4, color=COLORS[lbl], label=lbl)
axes[1].axvline(0, color="black", ls="--", lw=1.5)
axes[1].set_title("Net Profit Distribution — Counting Strategies")
axes[1].set_xlabel("Net Profit ($)"); axes[1].set_ylabel("Sessions"); axes[1].legend(fontsize=8)
plt.suptitle("Card Counting: Isolating Each Variable", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

Each step adds one ingredient:

- **Bet variation alone (no count): nothing.** −$45, identical to flat. Confirmed again.
- **Add Hi-Lo information, 6 decks: a real but partial repair.** Basic + Count 6D moves to −$12; full Counting 6D to −$5. The count identifies favourable shoes and the bet spread exploits them — but in a 6-deck shoe the swings are small, and the result lands just short of breakeven.
- **Single deck changes the game.** Basic + Count 1D jumps to **+$216**, full Counting 1D to **+$314**. Each card removed shifts the composition far more in one deck than in six, so favourable counts are more frequent and more extreme, and index deviations matter more.

The distribution overlay shows it physically: the 6-deck strategies sit centered near zero, while the single-deck strategies are shifted bodily to the right with long tails past +$3,000. Card counting on a single deck produces a real, measurable player edge — the headline result of the notebook. The practical asterisk is that single-deck games are rare, shuffle often, and carry worse base rules precisely to defend against this. And — as the risk section will show — that edge is far from free.

## How Counting Changes Decisions

What is the counter actually *doing* differently, hand to hand? Three mechanisms, one panel each: how it bets, why that's justified, and how it deviates from basic strategy.

In [ ]:
c6 = decisions["Counting + HiLo 6D"]
f6 = c6[c6.decision_index == 0].copy()
f6["tc"] = pd.cut(f6.true_count, bins=[-100, -1, 1, 3, 5, 100],
                  labels=["< -1", "-1 to 1", "1 to 3", "3 to 5", "> 5"])
bet_tc = f6.groupby("tc", observed=True).bet_size.mean()
win_tc = f6.groupby("tc", observed=True).is_win.mean() * 100

def hard16v10(lbl):
    d = decisions[lbl]
    m = ((d.decision_index == 0) & (d.player_value == 16) & (~d.player_is_soft)
         & (d.dealer_upcard == 10) & (~d.can_split) & (d.action.isin(["hit", "stand"])))
    return d[m]  # genuine hard 16: excludes 8-8 pairs and dealer-blackjack non-decisions
bc = hard16v10("Basic + Count + HiLo 6D"); cc = hard16v10("Counting + HiLo 6D")
def split_hs(d):
    v = d.action.astype(str).value_counts(normalize=True) * 100
    return [v.get("hit", 0), v.get("stand", 0)]
bvals, cvals = split_hs(bc), split_hs(cc)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(bet_tc.index.astype(str), bet_tc.values, color="#3F51B5", edgecolor="white")
axes[0].axhline(BASE_BET, color="black", ls="--", lw=1.5, label=f"Min bet (${BASE_BET})")
axes[0].set_title("Avg Bet Size by True Count\nCounting + HiLo 6D"); axes[0].set_xlabel("True Count"); axes[0].set_ylabel("Avg Bet ($)"); axes[0].legend()
for i, v in enumerate(bet_tc.values): axes[0].text(i, v + 0.4, f"${v:.0f}", ha="center", fontsize=9, fontweight="bold")

axes[1].bar(win_tc.index.astype(str), win_tc.values, color="#4CAF50", edgecolor="white")
axes[1].axhline(f6.is_win.mean() * 100, color="black", ls="--", lw=1.5, label="Overall avg")
axes[1].set_title("Win Rate by True Count\nCounting + HiLo 6D"); axes[1].set_xlabel("True Count"); axes[1].set_ylabel("Win Rate (%)"); axes[1].set_ylim(35, 55); axes[1].legend()
for i, v in enumerate(win_tc.values): axes[1].text(i, v + 0.2, f"{v:.1f}%", ha="center", fontsize=9, fontweight="bold")

x = np.arange(2); w = 0.35
axes[2].bar(x - w/2, bvals, w, label="Basic + Count", color="#4CAF50", edgecolor="white")
axes[2].bar(x + w/2, cvals, w, label="Counting", color="#3F51B5", edgecolor="white")
axes[2].set_xticks(x); axes[2].set_xticklabels(["Hit", "Stand"])
axes[2].set_title("Hard 16 (non-pair) vs Dealer 10 — Index Play"); axes[2].set_ylabel("% of decisions"); axes[2].legend()
axes[2].text(0.5, 0.93, f"Win rate: Basic {bc.is_win.mean()*100:.1f}% | Counting {cc.is_win.mean()*100:.1f}%",
             transform=axes[2].transAxes, ha="center", fontsize=9,
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
for i in range(2):
    axes[2].text(i - w/2, bvals[i] + 1, f"{bvals[i]:.0f}%", ha="center", fontsize=9)
    axes[2].text(i + w/2, cvals[i] + 1, f"{cvals[i]:.0f}%", ha="center", fontsize=9)
plt.suptitle("How Counting Changes Decisions at the Hand Level", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

**Bet size tracks the count, as designed.** At true counts below +1 the counter sits at the $10 minimum; from there bets climb to ~$19, ~$36, and ~$68 in the highest bucket — the 1-to-8 spread that approximates real counter behaviour without screaming at the pit boss.

**The win rate justifies it.** Win rate rises with the true count — from 43.1% at negative counts to 44.5% above +5. The gradient is modest (about a point and a half) but real and monotonic: a deck rich in tens and aces produces more blackjacks and more dealer busts. Betting more exactly when this edge appears is the whole mechanism — small per hand, but applied to the largest bets.

**Index plays change specific decisions.** Restricting to genuine hard 16 vs dealer 10 (excluding 8-8 pairs), basic strategy **always hits**. The counter instead **stands 47% of the time** — whenever the count says a ten-rich deck makes hitting too likely to bust. On this one hand the deviation lifts win rate from **20.5% to 23.2%**. Multiply that across thousands of such situations and it's a meaningful slice of the edge. Counting is information — tracked carefully and applied precisely — not a magic system.

## Risk of Ruin

Expected value is only half the story. A strategy that wins on average can still bankrupt you on the way — so the final question is how often each approach actually goes broke, and how widely outcomes spread.

In [ ]:
ror_order = [l for l in SESSION_RUNS if l != "Random + Flat"]
bust = [summary.loc[l, "bust"] for l in ror_order]
cols = [COLORS[l] for l in ror_order]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bars = axes[0].bar(range(len(ror_order)), bust, color=cols, edgecolor="white")
axes[0].set_xticks(range(len(ror_order)))
axes[0].set_xticklabels([l.replace(" + ", "\n+\n") for l in ror_order], fontsize=7)
axes[0].set_title("Ruin Rate by Strategy (% of sessions ending in bankruptcy)"); axes[0].set_ylabel("Ruin Rate (%)")
for b, v in zip(bars, bust):
    if v > 0: axes[0].text(b.get_x() + b.get_width()/2, v + 0.3, f"{v:.1f}%", ha="center", fontsize=8, fontweight="bold")

box_labels = ["Basic + Flat", "Basic + Martingale", "Counting + HiLo 1D"]
data = [sessions[l].ending_bankroll for l in box_labels]
bp = axes[1].boxplot(data, patch_artist=True, medianprops=dict(color="black", lw=2), showfliers=False)
for patch, l in zip(bp["boxes"], box_labels):
    patch.set_facecolor(COLORS[l]); patch.set_alpha(0.7)
axes[1].set_xticks(range(1, len(box_labels) + 1))
axes[1].set_xticklabels(["Basic\nFlat", "Basic\nMartingale", "Counting\nHiLo 1D"])
axes[1].axhline(START_BANKROLL, color="black", ls="--", lw=1.5, label=f"Start (${START_BANKROLL:,})")
axes[1].axhline(0, color="red", lw=1, label="Bust level")
axes[1].set_title("Ending Bankroll — Flat vs Martingale vs Counting"); axes[1].set_ylabel("Ending Bankroll ($)"); axes[1].legend(fontsize=9)
plt.suptitle("Risk of Ruin — How Often Does the Player Go Broke?", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

This is where the single-deck counting result earns its asterisk. **Flat betting is the safest** by far — 1.0% ruin rate — because $10 bets on a $1,000 bankroll can't be wiped out short of an extraordinary streak. **Martingale is the most dangerous**, 40.1% ruin, its doubling guaranteeing catastrophe in a long enough losing run.

The counterintuitive part: **single-deck counting is the highest-variance strategy here, not the lowest-risk.** Counting 1D pairs the best expected value in the study (+$314) with a **20.4% ruin rate** — one session in five ends in total bankruptcy. The boxplot shows why: its ending-bankroll spread is enormous. The same aggressive bet spread that manufactures the edge — $68 of a $1,000 bankroll at a high count — also means a cold streak at those bet sizes empties the account. The 6-deck counters go bankrupt far less (~6.5%), because their counts rarely climb high enough to trigger the biggest bets — but they barely clear breakeven either.

The lesson: **expected value and survival are independent levers.** Risk of ruin is governed by bet size relative to bankroll, not by decision quality. A flawless basic-strategy player goes bankrupt 40% of the time with Martingale and 1% flat; single-deck counting is wiped out 20% despite a genuine edge — all for the same reason. A counter who wanted that +$314 edge with flat-betting safety would simply need a far larger bankroll than $1,000.

## Conclusions

**1. The house edge is a long-run certainty, not a per-session one.** Optimal flat play loses $45 per 1,000-hand session on average, yet 44.8% of sessions finish in profit. The gap between certainty-in-aggregate and noise-in-the-moment is exactly what keeps players at the table.

**2. Betting systems move variance, not expectation.** Martingale and Anti-Martingale share flat betting's negative mean; what they add is a 37–40% chance of total ruin. And bet variation without information (count-betting with no count) is identical to flat — the bet pattern was never the point.

**3. Card counting works, but only under the right conditions.** In 6 decks it claws losses back to roughly breakeven (−$5 to −$12). In a single deck it flips to a real edge: +$314 per session, about +1.3% per dollar wagered — squarely in the range real single-deck counters achieve.

**4. That edge is not free — it is leveraged and high-variance.** The single-deck counter reaches +$314 by wagering 2.4× more and accepting a 20% ruin rate. Expected value and risk of ruin are separate questions; a positive edge does not imply safety, and on a $1,000 bankroll this edge carries real ruin risk.

**5. The simulation checks out against known mathematics.** Flat betting's edge per dollar wagered (−0.45%) matches the hand-level house edge exactly; counting raises win rate at high true counts as theory predicts; index plays produce the expected deviations. The results are mechanism, not coincidence.

---

*Next: the decision logs generated here train a neural network to learn this strategy from data alone — and the same session framework then evaluates the model head-to-head against these hand-coded strategies, on identical metrics.*